# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

# Likert Scale Analysis In-Depth

This notebook focuses on analyzing and visualizing survey data collected on a Likert scale (e.g., Strongly Disagree to Strongly Agree). We will cover reliability analysis and specialized visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('white')

## 1. Generate Synthetic Survey Data
We simulate a survey with 5 questions (items) administered to 50 respondents. The scale is 1-5 (1=Strongly Disagree, 5=Strongly Agree).

In [ ]:
np.random.seed(42)
n_respondents = 50
questions = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']

# Simulate responses where most people generally agree (skewed towards 4 and 5)
data = {}
for q in questions:
    data[q] = np.random.choice([1, 2, 3, 4, 5], size=n_respondents, p=[0.05, 0.1, 0.2, 0.4, 0.25])

df = pd.DataFrame(data)
print(df.head())

## 2. Reliability Analysis (Cronbach's Alpha)
Before aggregating Likert items into a single score, we must check if they measure the same underlying construct. Cronbach's Alpha is the standard metric (> 0.7 is generally acceptable).

In [ ]:
def cronbach_alpha(df):
    # 1. Transform the df into a correlation matrix
    df_corr = df.corr()
    
    # 2. Calculate N
    N = df.shape[1]
    
    # 3. Calculate the mean inter-item correlation
    rs = np.array([])
    for i, col in enumerate(df_corr.columns):
        sum_ = df_corr[col][i+1:].values
        rs = np.append(sum_, rs)
    mean_r = np.mean(rs)
    
    # 4. Use the formula
    cronbach_alpha = (N * mean_r) / (1 + (N - 1) * mean_r)
    return cronbach_alpha

alpha = cronbach_alpha(df)
print(f"Cronbach's Alpha: {alpha:.3f}")

if alpha > 0.7:
    print("The scale is reliable.")
else:
    print("The scale may not be reliable.")

## 3. Visualization: Diverging Stacked Bar Chart
The standard for visualizing Likert scales is the Diverging Stacked Bar Chart. It centers the neutral option, allowing easy comparison of positive vs. negative sentiment.

In [ ]:
# Prepare data for plotting
likert_counts = df.apply(pd.Series.value_counts).fillna(0).T

# Ensure all columns 1-5 exist
for i in range(1, 6):
    if i not in likert_counts.columns:
        likert_counts[i] = 0
likert_counts = likert_counts[[1, 2, 3, 4, 5]]

# Convert to percentages
likert_pct = likert_counts.div(likert_counts.sum(axis=1), axis=0) * 100

# Create the plot
ax = likert_pct.plot(kind='barh', stacked=True, figsize=(10, 6), 
                     colormap='RdYlGn', edgecolor='none')

# Move the legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Response')
plt.title('Likert Scale Responses (Percentage)')
plt.xlabel('Percentage')
plt.ylabel('Question')

# Add a vertical line at 50% (though true diverging charts center the neutral category, this is a standard stacked approx)
plt.axvline(50, color='black', linestyle='--', alpha=0.5)

plt.show()

## 4. Summary Statistics
While mean is controversial for ordinal data, mode and median are safe. We'll calculate all three.

In [ ]:
summary = pd.DataFrame({
    'Mean': df.mean(),
    'Median': df.median(),
    'Mode': df.mode().iloc[0]
})
print(summary)